# Aero-RAG — a walkthroughAsking questions about real aviation incidents, grounded in the source reports.This notebook narrates the pipeline end to end, reusing the modules in `src/`.It's the companion to the Medium article. Run it from the repo root.

## 1. Load the reportsWe keep only the free-text narrative and give each report an id (so the model can cite its sources).

In [ ]:
import sys; sys.path.append("..")from src.data import load_reportsdf = load_reports(sample_size=4000)df.head(3)

## 2. Why RAG?A language model alone would answer from its parametric memory — fluent, but withno guarantee it reflects *these* reports, and no way to trace a claim back to asource. RAG grounds every answer in retrieved text. Two phases:- **Offline:** split reports into chunks, embed them, store them in a vector DB.- **Online:** embed the question, retrieve the closest chunks, feed them to the LLM.

## 3. Build the index (offline)Chunk every report, embed the chunks, and persist them in ChromaDB. Run once.

In [ ]:
import torchfrom src.ingest import build_indexdevice = "cuda" if torch.cuda.is_available() else "cpu"n_chunks = build_index(df, device=device)print(n_chunks, "chunks indexed")

## 4. Retrieval, on its ownBefore generating anything, let's see what retrieval returns for a question.These are the raw chunks the LLM will be allowed to use — nothing else.

In [ ]:
from src.rag import RAGPipelinerag = RAGPipeline()          # loads embedder + index + local LLMdocs, metas = rag.retrieve("What are common causes of altitude deviations?", k=4)for d, m in zip(docs, metas):    print(f"[{m['doc_id']}] {d[:200]}...\n")

## 5. Generation, groundedNow the full loop: the retrieved chunks are injected into the prompt, and themodel is instructed to answer *only* from them (and to say "I don't know"otherwise). The answer comes back with its source report ids.

In [ ]:
result = rag.answer("What are common causes of altitude deviations?")print(result.answer)print("\nSources:", result.sources)

## 6. Is the answer grounded?Generation is hard to evaluate. A cheap, useful signal is **groundedness**: thesimilarity between the answer and the retrieved context. A low score means themodel may have drifted away from the sources — a flag for human review.

In [ ]:
from src.evaluate import groundednessscore = groundedness(result.answer, result.contexts)print(f"Groundedness: {score:.3f}")

## 7. A few more questions

In [ ]:
for q in [    "How do crews handle an engine failure on departure?",    "What role does fatigue play in reported incidents?",    "Who won the 2018 World Cup?",          # out-of-scope: should say 'I don't know']:    r = rag.answer(q)    print("Q:", q)    print("A:", r.answer)    print("Sources:", r.sources, "\n")

## Takeaways- RAG turns an ungrounded LLM into one that answers from a real, citable corpus.- The out-of-scope question shows the guardrail working: no context, no answer.- Reliability isn't free — groundedness scoring and a human in the loop matter,  especially in a safety domain like aviation.See the README for tuning (model size, retrieval depth) and next steps.